## Interaction Embedding

This notebook produces a fixed-size vector representation for each student–exercise interaction in the MIAAM dataset.

**Pipeline:**
1. Load `maths_data_filtered.parquet` (student attempt records).
2. Filter out interactions whose exercise has no compressed screenshot, so that all remaining records have visual content available.
3. Map string `user_id` values to contiguous integer indices (`user_id_int`).
4. Encode interactions in chunks of 1 000 000 rows using `encode_interactions` from `attempt_embedding` — each interaction is represented as a `Float32` vector derived from the attempt context.
5. Save each chunk to `data/interactions_embedded_part_{i}.parquet` with a globally-unique `interaction_id` offset applied across chunks.

**Output:** one embedding per interaction, saved across multiple part files, suitable as sequential input features for downstream student-modelling or knowledge-tracing models.

In [1]:

import polars as pl
import json
import pathlib
import os
import numpy as np
from exercise_embedding import encode_all
from attempt_embedding import encode_interactions
from tqdm import tqdm
import gc
import torch



DATASET = pathlib.Path("../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")

/home/guglielmo/Projects/stundetsTracker/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
interactions.shape

(6481693, 14)

In [4]:
interactions.head()

user_id,classroom_id,playlist_or_module_id,exercise_id,created_at,data_correct,work_mode,data_answer,data_duration,source,attempt_index,session_id,module_id,module_name
str,str,str,str,datetime[μs],bool,str,str,f64,str,u32,str,str,str
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""97516718-fd8f-4f90-8ead-56dff6…",2023-12-01 00:00:12.815,true,"""zpdes""","""[1]""",2335.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""f90ab87b-7054-4de8-b836-300653…",2023-11-24 00:01:50.092,true,"""zpdes""","""[1]""",3508.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""f90ab87b-7054-4de8-b836-300653…",2023-11-24 00:01:50.092,true,"""zpdes""","""[1]""",3508.0,"""am""",2,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""21a30e28-d557-4760-a3dd-3838f4…",2023-11-24 00:06:36.815,true,"""zpdes""","""[0]""",1842.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""21a30e28-d557-4760-a3dd-3838f4…",2023-11-24 00:06:36.815,true,"""zpdes""","""[0]""",1842.0,"""am""",2,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""


In [6]:
# 1. Interactions per student, sorted
counts = (
    interactions.group_by("user_id")
      .len()
      .sort("len", descending=True)
)
print(counts.head(20))

shape: (20, 2)
┌─────────────────────────────────┬──────┐
│ user_id                         ┆ len  │
│ ---                             ┆ ---  │
│ str                             ┆ u32  │
╞═════════════════════════════════╪══════╡
│ dde29c97-ec1e-4dbe-b6fa-0e364a… ┆ 5158 │
│ cc06a716-7731-4e6d-96ba-fd8bdb… ┆ 3830 │
│ a552d727-5161-4288-9737-ae9f04… ┆ 3807 │
│ b8fbe4bf-8943-4d1a-9cc6-d07479… ┆ 3700 │
│ 2abfbff1-6258-4adb-8282-ee425a… ┆ 3685 │
│ …                               ┆ …    │
│ e6f84372-6de8-4773-b0d0-682391… ┆ 2768 │
│ 633bc53d-4a9e-46d1-a68a-9055f4… ┆ 2711 │
│ 492d5ba0-bbf4-417c-aec1-fd5294… ┆ 2657 │
│ f44adc41-dc56-4537-802d-133e3c… ┆ 2561 │
│ bf671167-6d9d-4c94-947b-98b8bd… ┆ 2515 │
└─────────────────────────────────┴──────┘


In [7]:
print(counts["len"].describe())
print(counts.filter(pl.col("len") > 512).height, "students over 512")
print(counts.filter(pl.col("len") > 1024).height, "students over 1024")

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 38520.0   │
│ null_count ┆ 0.0       │
│ mean       ┆ 168.26825 │
│ std        ┆ 243.51083 │
│ min        ┆ 5.0       │
│ 25%        ┆ 37.0      │
│ 50%        ┆ 86.0      │
│ 75%        ┆ 192.0     │
│ max        ┆ 5158.0    │
└────────────┴───────────┘
2693 students over 512
623 students over 1024


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [7]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"\nAfter filtering: {interactions.shape[0]} attempts, {interactions['exercise_id'].n_unique()} unique exercises")


After filtering: 6263671 attempts, 6936 unique exercises


In [8]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)
interactions = interactions.join(user_id_map, on="user_id", how="left")
del user_id_map

In [9]:

CHUNK_SIZE = 1000000
n_rows = len(interactions)
n_chunks = (n_rows + CHUNK_SIZE - 1) // CHUNK_SIZE

part_paths = []

for i in range(n_chunks):
    part_path = SAVE_FOLDER / f"interactions_embedded_part_{i}.parquet"

    if part_path.exists():
        print(f"Part {i+1}/{n_chunks} already exists, skipping.")
        part_paths.append(part_path)
        continue

    start = i * CHUNK_SIZE
    chunk = interactions.slice(start, CHUNK_SIZE)
    print(f"Encoding part {i+1}/{n_chunks} (rows {start}–{start + len(chunk) - 1})...")

    chunk_embeddings = encode_interactions(chunk)
    # Offset so interaction_id is globally unique across all chunks
    chunk_embeddings = chunk_embeddings.with_columns(
        (pl.col("interaction_id") + start).alias("interaction_id")
    )
    chunk_embeddings.write_parquet(part_path)
    part_paths.append(part_path)

    del chunk, chunk_embeddings
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  Saved and freed part {i+1}.")

print(f"Done — {n_chunks} parts saved to {SAVE_FOLDER}")

Part 1/7 already exists, skipping.
Part 2/7 already exists, skipping.
Part 3/7 already exists, skipping.
Part 4/7 already exists, skipping.
Part 5/7 already exists, skipping.
Part 6/7 already exists, skipping.
Part 7/7 already exists, skipping.
Done — 7 parts saved to ../data
